# 3.1 · El mismo agente, en una fracción del código

**Objetivo:** reescribir con el Agent SDK el agente del 2.2 y **ver desaparecer** el código frágil.

**Al terminar tendrás:** un agente que resuelve la misma pregunta del 2.2, sin bucle escrito a mano.

> Abre `2.2-bucle-agente.ipynb` a la derecha (**View → Editor Layout → Split Right**): al final los compararemos.

## Preparación

Ya no necesitamos `anthropic.Anthropic()`: el SDK se encarga de las llamadas a la API.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

from herramientas import restaurar_datos

## Lo que vamos a borrar

Las piezas frágiles de `ejecutar_agente()` del 2.2. Todas las hemos escrito y casi todas las hemos roto:

```python
for i in range(MAX_ITERACIONES):                                   # ← límite de iteraciones
    response = client.messages.create(...)
    messages.append({"role": "assistant", "content": response.content})   # ← orden del historial
    if response.stop_reason != "tool_use":                         # ← condición de parada
        return ...
    for bloque in response.content:
        try:
            salida = FUNCIONES[bloque.name](**bloque.input)
            resultados.append({"type": "tool_result", "tool_use_id": bloque.id, ...})   # ← ids
        except Exception as e:
            resultados.append({..., "is_error": True})            # ← errores
    messages.append({"role": "user", "content": resultados})
```

## Portar las herramientas

En el SDK, una herramienta propia se declara con el decorador `@tool(nombre, descripción, esquema)` sobre una función `async` que recibe un diccionario con los argumentos y devuelve el resultado en el formato de bloques de contenido. En el bloque 4 lo veremos a fondo.

**Las funciones de negocio no cambian:** las importamos de `herramientas.py` con otro nombre y solo cambia la forma de declararlas.

El esquema más corto es un mapa de nombre → tipo de Python. Ojo: con esta forma **todos los campos son obligatorios**, así que el modelo enviará un string vacío en los filtros que no quiera usar (`buscar_incidencias` los ignora). En el 4.1 lo haremos mejor con JSON Schema.

In [ ]:
import json
from claude_agent_sdk import tool, create_sdk_mcp_server
from herramientas import buscar_incidencias as _buscar, crear_incidencia as _crear

@tool("buscar_incidencias",
      "Busca incidencias de soporte existentes por texto, estado o cliente. Devuelve una lista JSON.",
      {"texto": str, "estado": str, "cliente": str})
async def buscar_incidencias(args):
    return {"content": [{"type": "text", "text": json.dumps(_buscar(**args), ensure_ascii=False)}]}

@tool("crear_incidencia",
      "Crea una incidencia nueva. Úsala solo si no existe ya una abierta sobre el mismo problema.",
      {"cliente": str, "asunto": str, "descripcion": str, "prioridad": str})
async def crear_incidencia(args):
    return {"content": [{"type": "text", "text": json.dumps(_crear(**args), ensure_ascii=False)}]}

Una ventaja del notebook: cada herramienta se puede probar sola antes de dársela al agente. `.handler` es la función original:

In [ ]:
await buscar_incidencias.handler({"texto": "", "estado": "abierta", "cliente": "Initech"})

Las herramientas se agrupan en un **servidor MCP**. MCP (*Model Context Protocol*) es el estándar con el que el SDK habla con cualquier herramienta. Este servidor es *in-process*: corre dentro del propio kernel.

In [ ]:
soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[buscar_incidencias, crear_incidencia])

## El agente

El mismo system prompt y la misma pregunta del 2.2.

In [ ]:
SYSTEM_TEXTO = (
    "Eres un agente de soporte de nivel 1 de una empresa de software. "
    "Respondes en español, de forma breve y basándote solo en los datos que tengas."
)

PREGUNTA = (
    "El cliente Globex dice que no puede exportar a PDF. Si ya hay una incidencia abierta "
    "sobre eso, dime cuál; si no, créala con prioridad media."
)

Toda la configuración del agente va en `ClaudeAgentOptions`:

- `mcp_servers`: los servidores de herramientas, con el nombre que les damos.
- `tools=[]`: el SDK trae sus propias herramientas integradas (leer y escribir ficheros, ejecutar comandos…). Este agente no las necesita, así que no le damos ninguna. Lo que no tiene, no lo puede usar, y además cada herramienta que se le describe al modelo ocupa tokens en cada llamada.
- `allowed_tools`: las herramientas que el agente puede usar **sin pedir permiso**.
- `max_turns`: el límite de iteraciones (nuestro `MAX_ITERACIONES`).
- `setting_sources=[]`: que no cargue ninguna configuración local de Claude del disco. Lo veremos en el 3.2.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage, ToolUseBlock, TextBlock

options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM_TEXTO,
    mcp_servers={"soporte": soporte},
    tools=[],   # sin herramientas integradas (Read, Bash…): solo las nuestras
    allowed_tools=["mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia"],
    max_turns=10,
    setting_sources=[],
)

Una función auxiliar para imprimir lo que hace el agente. La reutilizaremos en el resto del taller.

In [ ]:
def imprimir(msg):
    if isinstance(msg, AssistantMessage):
        for b in msg.content:
            if isinstance(b, ToolUseBlock):
                print(f"  → {b.name}({b.input})")
            elif isinstance(b, TextBlock):
                print(b.text)
    elif isinstance(msg, ResultMessage):
        print(f"[{msg.subtype}] turnos={msg.num_turns} coste=${msg.total_cost_usd:.4f}")

`query()` lanza el agente y devuelve los mensajes **a medida que se producen**. El SDK es asíncrono: en Jupyter se usa `async for` directamente en la celda.

In [ ]:
restaurar_datos()

In [ ]:
mensajes_sdk = []
async for msg in query(prompt=PREGUNTA, options=options):
    mensajes_sdk.append(msg)
    imprimir(msg)

El mismo comportamiento que en el 2.2: busca, no encuentra, crea.

### Qué mensajes emite el SDK

In [ ]:
[type(m).__name__ for m in mensajes_sdk]

| Mensaje | Qué es | En el 2.2 era… |
|---|---|---|
| `SystemMessage` | inicio de la sesión: modelo, herramientas disponibles… | — |
| `AssistantMessage` | lo que dice o pide el modelo (`TextBlock`, `ToolUseBlock`) | `{"role": "assistant", ...}` |
| `UserMessage` | los resultados de las herramientas (`ToolResultBlock`) | `{"role": "user", "content": [tool_result]}` |
| `ResultMessage` | el final: resultado, turnos, coste | el `return` del bucle |

**Es la misma secuencia que `historial` en el 2.2.**

In [ ]:
mensajes_sdk[-1]

### El nombre de las herramientas

Fíjate en las trazas: las herramientas se llaman `mcp__soporte__buscar_incidencias`. El formato es siempre `mcp__<servidor>__<herramienta>`, y es el nombre que hay que usar en `allowed_tools`. Aparecerá constantemente.

### `asyncio.run` no funciona en Jupyter

En un script, el código asíncrono se lanza con `asyncio.run(main())`. En Jupyter falla, porque el notebook ya tiene un *event loop* en marcha:

In [ ]:
# ❌ FALLO PROVOCADO
import asyncio

async def main():
    async for msg in query(prompt=PREGUNTA, options=options):
        imprimir(msg)

asyncio.run(main())

En el notebook: `await main()` o `async for` directamente en la celda.

## El diff

Pon el notebook 2.2 a la izquierda y este a la derecha. Qué ha pasado con cada pieza frágil:

| En el bucle manual | En el SDK |
|---|---|
| `while` + `stop_reason` | lo hace `query()` |
| `append` del assistant / tool_result | gestionado por el SDK |
| `tool_use_id` | gestionado por el SDK |
| `try/except` → `is_error` | el SDK devuelve el error al modelo |
| `MAX_ITERACIONES` | `max_turns=10` |
| `usage` sumado a mano | `ResultMessage.total_cost_usd` |

No hay magia: por dentro está **exactamente** el bucle que escribimos. Lo que gana el SDK es que ese bucle ya está probado y endurecido.

## Resumen

- **Lo que ganamos:** desaparece el código frágil del bucle. Nos quedan las herramientas, el prompt y las opciones.
- **Lo que cedemos:** el control fino del bucle. La API directa sigue teniendo sentido en flujos muy controlados, sin herramientas de sistema, o cuando la latencia es crítica.
- El SDK es asíncrono: en Jupyter, `async for` / `await` directamente en la celda, nunca `asyncio.run`.
- Las herramientas se llaman `mcp__<servidor>__<herramienta>`.

## 🔸 Reto

Porta también `cerrar_incidencia` (el reto del 2.2) y pídele al agente que cierre las incidencias de baja prioridad de ACME.

### Solución

La herramienta, un servidor nuevo con las tres y unas opciones que la permitan. `options` se sustituye entera: `ClaudeAgentOptions` no se modifica en sitio.

In [ ]:
from herramientas import cerrar_incidencia as _cerrar

@tool("cerrar_incidencia",
      "Cierra una incidencia abierta por su id (formato INC-123). Si no conoces el id, búscalo antes.",
      {"id": str})
async def cerrar_incidencia(args):
    return {"content": [{"type": "text", "text": json.dumps(_cerrar(args["id"]), ensure_ascii=False)}]}

soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[buscar_incidencias, crear_incidencia, cerrar_incidencia])

options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM_TEXTO,
    mcp_servers={"soporte": soporte},
    tools=[],
    allowed_tools=["mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia",
                   "mcp__soporte__cerrar_incidencia"],
    max_turns=10,
    setting_sources=[],
)

In [ ]:
restaurar_datos()
async for msg in query(prompt="Cierra las incidencias abiertas de baja prioridad de ACME.", options=options):
    imprimir(msg)

In [ ]:
from herramientas import buscar_incidencias as _buscar
[(i["id"], i["prioridad"], i["estado"]) for i in _buscar(cliente="ACME")]

In [ ]:
restaurar_datos()